# Start here — frozen La Palma ML-FCUC

This notebook reads the **actual shipped numerical inputs**, checks the source/cut identities, and prepares output folders. It does not fit a classifier, integrate the production library, or launch a solver.

## The three-file idea
`START_LAPALMA.gms` chooses the experiment; `Input_Data_LaPalma_Weekly.gms` binds the `.inc` numerical tables to symbols; `LaPalma_FCUC_35Hz.gms` defines and solves the mixed-integer problem. The affine frequency-security coefficients are already supplied in `inc/LP35_Calibrated.inc`. A new reader should not retrain a classifier before they can solve a single example.

## Inputs, dimensions and units
There are 11 units, 24 hourly periods, 3 dispatch blocks and 8 startup categories. Four independent seasonal weeks give 672 hourly input points. The source's `transmission_option=4` selects column `tr4`; it does not activate a network model. Power/headroom are in MW, inertia energy is in MW s, and gain is in MW/Hz. The optimizer minimizes kEUR; EUR outputs multiply by 1000.

Run **Kernel → Restart Kernel and Run All Cells**. All dependencies of this notebook are inside the repository plus the declared Python libraries. Missing `LP_Case.json` and `LP35_Protocol.json` affect the full research replay, not this frozen-deployment route.

In [1]:
from pathlib import Path
import json, hashlib, re
import numpy as np
import pandas as pd


def find_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'GAMS_LaPalma' / 'START_LAPALMA.gms').is_file():
            return candidate
    raise FileNotFoundError('Extract the complete repository, then open this notebook inside it.')


def read_inc_table(path, row_dimensions):
    """Read the shipped XLS2GMS numerical tables; never evaluate source code.

    Leading star-comments and blank padding are ignored. Dotted row tuples,
    e.g. i1 . b1, identify row dimensions; the first data line is the header.
    The function intentionally supports this repository's tables, not all GAMS.
    """
    lines = [s.strip() for s in Path(path).read_text(encoding='utf-8-sig').splitlines()
             if s.strip() and not s.lstrip().startswith('*')]
    if not lines:
        raise ValueError(f'Empty table: {path}')
    columns = lines[0].split()
    records, labels = [], []
    for line_number, line in enumerate(lines[1:], start=2):
        parts = [x for x in line.split() if x != '.']
        keys, fields = parts[:row_dimensions], parts[row_dimensions:]
        if len(keys) != row_dimensions or len(fields) != len(columns):
            raise ValueError(f'Malformed table row {line_number} in {path}')
        values = [0.0 if v.lower() == 'eps' else float(v) for v in fields]
        if not np.isfinite(values).all():
            raise ValueError(f'Non-finite table values in {path}')
        labels.append(keys[0] if row_dimensions == 1 else tuple(keys))
        records.append(values)
    index = pd.Index(labels) if row_dimensions == 1 else pd.MultiIndex.from_tuples(labels)
    if not index.is_unique:
        raise ValueError(f'Duplicate row identifiers in {path}')
    return pd.DataFrame(records, index=index, columns=columns)


def load_inputs(root):
    inc = Path(root) / 'GAMS_LaPalma' / 'inc'
    ids = [f'i{i}' for i in range(1, 12)]
    def vector(name, col):
        frame = read_inc_table(inc / name, 1)
        return frame.loc[ids, col].to_numpy(dtype=float)
    blocks = read_inc_table(inc / 'block_max.inc', 1).loc[ids, ['b1','b2','b3']]
    units = pd.DataFrame({'unit': ids, 'pmax_mw': blocks.sum(axis=1).to_numpy(),
        'pmin_mw': vector('aux9.inc','tr4'), 'inertia_s': vector('auxinertia.inc','column1'),
        'mbase': vector('auxmbase.inc','column1'), 'gain_pu': vector('auxk.inc','column1'),
        'no_load_keur_h': vector('aux4.inc','tr4'), 'ramp_up_mw_h': vector('aux5.inc','tr4'),
        'ramp_down_mw_h': vector('aux6.inc','tr4'), 'min_up_h': vector('aux8.inc','tr4'),
        'min_down_h': vector('aux7.inc','tr4'), 'initial_on_age_h': vector('aux3.inc','column1'),
        'initial_off_age_h': vector('aux2.inc','column1'), 'initial_power_mw': vector('aux10.inc','column1')})
    units['initial_commitment'] = (units.initial_on_age_h > 0).astype(int)
    units['energy_mws'] = units.inertia_s * units.mbase
    units['gain_mw_hz'] = units.gain_pu * units.mbase / 50.0
    demand = read_inc_table(inc / 'd_seasons.inc', 2)
    wind = read_inc_table(inc / 'RES_seasons.inc', 2)
    profiles = []
    for season in ['winter','spring','summer','autumn']:
        for day in range(1,8):
            for hour in range(1,25):
                key=(f't{hour}', f'd{day}')
                profiles.append({'season':season, 'day':day, 'hour':hour,
                    'demand_mw':float(demand.loc[key,season]), 'wind_available_mw':float(wind.loc[key,season])})
    return units, pd.DataFrame(profiles), blocks

ROOT=find_root()
UNITS,PROFILES,BLOCKS=load_inputs(ROOT)
DEP=json.loads((ROOT/'GAMS_LaPalma/LP35_Deployment.json').read_text())
assert DEP['campaign']=='LP35_6259c70d0303'
assert DEP['model_id']=='0899f2bc59cecb10'
assert DEP['nadir_min_hz']==46.5 and DEP['nadir_drop_hz']==3.5
assert len(UNITS)==11 and len(PROFILES)==672
print('PASS: frozen identity, eleven units, four complete 168-hour profiles.')


PASS: frozen identity, eleven units, four complete 168-hour profiles.


## 1. Check source preservation, not only filenames
A file can exist and still contain the wrong experiment. The two frozen generated includes and `cplex.opt` are checked byte-for-byte. Numerical tables and documented GAMS files are checked after removing full-line comments and whitespace; this permits removal of a private workbook-location comment, not a changed numerical token.

The author-uploaded originals were separately checked against their frozen digests during packaging. Their exact bytes are retained in the private backup. A sanitized public table is intentionally not advertised as the original workbook export byte stream.

In [2]:
def normalized_gams(text):
    lines=[]; intext=False
    for line in text.splitlines():
        if line.lstrip().lower().startswith('$ontext'):
            intext=True; continue
        if line.lstrip().lower().startswith('$offtext'):
            intext=False; continue
        if not intext and not line.lstrip().startswith('*'):
            lines.append(line)
    return re.sub(r'\s+','','\n'.join(lines))

records=json.loads((ROOT/'verification/source_comparison.json').read_text())
for record in records:
    path=ROOT/record['path']
    actual=hashlib.sha256(normalized_gams(path.read_text()).encode()).hexdigest()
    assert actual==record['noncomment_sha256'], f"Changed scientific content: {record['path']}"
for name,expected in [('inc/LP35_Calibrated.inc',DEP['cut_sha256']),
                      ('inc/LP35_Jobs.inc',DEP['jobs_sha256']),
                      ('cplex.opt',DEP['sources']['GAMS_LaPalma/cplex.opt'])]:
    assert hashlib.sha256((ROOT/'GAMS_LaPalma'/name).read_bytes()).hexdigest()==expected, name
required=re.findall(r'^\$include\s+(inc/\S+)',(ROOT/'GAMS_LaPalma/Input_Data_LaPalma_Weekly.gms').read_text(),re.M)
assert len(required)==18 and all((ROOT/'GAMS_LaPalma'/p).is_file() for p in required)
assert (ROOT/'GAMS_LaPalma/cplex.opt').read_text().strip()=='threads 4'
print(f'PASS: {len(records)} scientific-source comparisons; 18/18 primary numerical includes; frozen exports and CPLEX option file.')


PASS: 33 scientific-source comparisons; 18/18 primary numerical includes; frozen exports and CPLEX option file.


## 2. Read the real unit and seasonal data
For unit $i$, `energy_mws = inertia_s × mbase` and `gain_mw_hz = gain_pu × mbase / 50`. The original initial condition has all units off for 24 hours and zero incoming dispatch. It is an incoming UC state, not a feasible operating schedule supplying demand.

The following consistency checks concern input dimensions and bounds. They do not establish optimality or frequency security of a schedule that has not yet been solved.

In [3]:
assert np.all(UNITS.pmax_mw>=UNITS.pmin_mw)
assert np.all(UNITS.pmin_mw>=0) and np.all(UNITS.energy_mws>0)
assert np.all(PROFILES.demand_mw>=0) and np.all(PROFILES.wind_available_mw>=0)
assert np.allclose(UNITS.energy_mws,UNITS.inertia_s*UNITS.mbase)
assert np.allclose(50*UNITS.gain_mw_hz,UNITS.gain_pu*UNITS.mbase)
assert np.all(UNITS.initial_commitment==0)
display(UNITS.round(6))
display(PROFILES.groupby('season',sort=False).agg(hours=('hour','size'),
    demand_min_mw=('demand_mw','min'),demand_max_mw=('demand_mw','max'),
    demand_energy_mwh=('demand_mw','sum'),wind_available_mwh=('wind_available_mw','sum')))


,unit,pmax_mw,pmin_mw,inertia_s,mbase,gain_pu,no_load_keur_h,ramp_up_mw_h,ramp_down_mw_h,min_up_h,min_down_h,initial_on_age_h,initial_off_age_h,initial_power_mw,initial_commitment,energy_mws,gain_mw_hz
0,i1,3.82,2.35,1.749,5.40,20.000000,0.070203,3.82,3.82,1.0,1.0,0.0,24.0,0.0,0,9.4446,2.16
1,i2,3.82,2.35,1.749,5.40,20.000000,0.070203,3.82,3.82,1.0,1.0,0.0,24.0,0.0,0,9.4446,2.16
2,i3,3.82,2.35,1.749,5.40,20.000000,0.070203,3.82,3.82,1.0,1.0,0.0,24.0,0.0,0,9.4446,2.16
3,i4,4.30,2.82,1.734,6.30,20.000000,0.101700,4.30,4.30,1.0,1.0,0.0,24.0,0.0,0,10.9242,2.52
4,i5,6.70,3.30,2.161,9.40,20.000000,0.189613,6.70,6.70,1.0,1.0,0.0,24.0,0.0,0,20.3134,3.76
5,i6,6.70,3.30,1.878,9.60,20.000000,0.189613,6.70,6.70,1.0,1.0,0.0,24.0,0.0,0,18.0288,3.84
6,i7,11.20,6.63,2.100,15.75,20.000000,0.161812,11.20,11.20,1.0,1.0,0.0,24.0,0.0,0,33.0750,6.30
7,i8,11.50,6.63,2.100,14.50,20.000000,0.169909,11.50,11.50,1.0,1.0,0.0,24.0,0.0,0,30.4500,5.80
8,i9,11.50,6.63,2.100,14.50,20.000000,0.169909,11.50,11.50,1.0,1.0,0.0,24.0,0.0,0,30.4500,5.80
9,i10,11.50,6.63,2.100,14.50,20.000000,0.169909,11.50,11.50,1.0,1.0,0.0,24.0,0.0,0,30.4500,5.80


,hours,demand_min_mw,demand_max_mw,demand_energy_mwh,wind_available_mwh
season,,,,,
winter,168,18.204065,36.196543,4593.609161,442.304778
spring,168,18.140493,33.929003,4465.673055,459.814409
summer,168,19.961293,35.252787,4752.710768,719.922350
autumn,168,18.697109,35.901473,4658.500459,548.023676


## 3. Prepare folders — this does not run the campaign
The original training/export notebook created the output folders before the GAMS launcher. The public release makes this dependency explicit. Creating 336 empty daily directories is cheap and does not authorize 336 solves. The GAMS start file can also create them on Windows.

The archived deployment manifest contains the original **1.25% / 600 s** plan. The uploaded working GAMS source uses **0.5% / 300 s** with a 1,200 s search limit; `cplex.opt` specifies only four threads. The separate run-configuration record describes this difference without editing either scientific manifest.

In [4]:
CREATE_OUTPUT_DIRECTORIES=True
count=0
if CREATE_OUTPUT_DIRECTORIES:
    for run in DEP['runs']:
        for season in DEP['seasons']:
            for day in range(1,8):
                (ROOT/'GAMS_LaPalma/Results_35Hz'/DEP['campaign']/run['run']/season/f'd{day}').mkdir(parents=True,exist_ok=True)
                count+=1
assert count==336 if CREATE_OUTPUT_DIRECTORIES else count==0
print(f'Prepared {count} empty-or-existing daily output locations. No files overwritten; no solve launched.')
config=json.loads((ROOT/'verification/Run_Configuration.json').read_text())
display(pd.DataFrame([config['uploaded_working_defaults']]))
report={'status':'PASS','source_comparisons':len(records),'primary_numerical_includes':len(required),
        'units':len(UNITS),'hourly_input_points':len(PROFILES),'folders_prepared':count,'native_gams_execution':'NOT RUN'}
out=ROOT/'verification/local_reports';out.mkdir(parents=True,exist_ok=True)
(out/'00_quick_start.json').write_text(json.dumps(report,indent=2)+'\n')
print('Next: open GAMS_LaPalma/START_LAPALMA.gms in the GAMS IDE; leave SCOPE=DAY for the first native example.')


Prepared 336 empty-or-existing daily output locations. No files overwritten; no solve launched.


,search_seconds,search_gap,polish_seconds,polish_gap,threads
0,1200,0.005,300,0.0001,4


Next: open GAMS_LaPalma/START_LAPALMA.gms in the GAMS IDE; leave SCOPE=DAY for the first native example.


## 4. Native execution and expected output
In the **GAMS 24.9 IDE**, use `GAMS_LaPalma` as the project working directory and open `START_LAPALMA.gms`. The default is `LR_C1`, `winter`, day 1. Press the IDE Run button/F9. An accepted solve produces `results.gdx`, `state.gdx`, `Results_LR_C1_winter_d1.xlsx`, and `COMPLETE.txt` under the corresponding daily folder.

A small relative gap is not the primal-feasibility test. The model retains the original incumbent policy, fixes commitment for polishing, checks equation/bound/integrality residuals, and only then exports accepted state. Export-only recovery never re-solves a completed, identity-matched GDX state. Read `RUN_LAPALMA.md` for the WEEK and ALL choices.

GAMS/CPLEX and Windows Excel-export components are external runtime requirements. They were not available in the build environment; native optimization is **NOT RUN**, not a claimed PASS.